# Compare DL Models — Bihar Flood Prediction
This notebook compares all 5 DL models:
- Deep ANN
- 1D-CNN
- Bidirectional LSTM
- Autoencoder + Classifier
- TabNet

**Run all individual model notebooks first**, then run this notebook to aggregate results.

Alternatively, you can run all models inline here using Section B.

## Option A — Load Saved Results (if you ran individual notebooks)
Each model saves a `dl_*_results.json` file. Load them here.

In [ ]:
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

result_files = [
    'dl_ann_results.json',
    'dl_cnn_results.json',
    'dl_lstm_results.json',
    'dl_autoencoder_results.json',
    'dl_tabnet_results.json',
]

results = []
for f in result_files:
    if os.path.exists(f):
        with open(f) as fp:
            results.append(json.load(fp))
    else:
        print(f'WARNING: {f} not found — run the corresponding model notebook first.')

if results:
    df_results = pd.DataFrame(results)
    df_results = df_results.sort_values('f1', ascending=False).reset_index(drop=True)
    print(df_results.to_string(index=False))
else:
    print('No result files found. Use Option B below to train all models inline.')

## Option B — Train All Models Inline
Run this section if you want to train and compare everything in one notebook.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report, roc_auc_score
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Dense, Dropout, BatchNormalization, Input,
    Conv1D, MaxPooling1D, Flatten,
    LSTM, Bidirectional
)
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

np.random.seed(42)
tf.random.set_seed(42)

# ── Data Prep ──────────────────────────────────────────────────
df = pd.read_csv('bihar_flood_data.csv')
df['Date'] = pd.to_datetime(df['Date'])
df['year']       = df['Date'].dt.year
df['month']      = df['Date'].dt.month
df['day']        = df['Date'].dt.day
df['is_monsoon'] = df['month'].apply(lambda x: 1 if x in [6,7,8,9] else 0)
df.drop(columns=['Date'], inplace=True)
df = pd.get_dummies(df, columns=['District'], drop_first=True)

X = df.drop(columns=['Flood']).values
y = df['Flood'].values

scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

classes = np.unique(y_train)
wts = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight = dict(zip(classes, wts))

input_dim = X_train.shape[1]
es = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

results_inline = []  # Store all model metrics here

def evaluate(name, model, X_te, y_te, reshape=None):
    X_in = X_te.reshape(*reshape) if reshape else X_te
    prob = model.predict(X_in, verbose=0).flatten()
    pred = (prob >= 0.5).astype(int)
    acc  = accuracy_score(y_te, pred)
    f1   = f1_score(y_te, pred)
    auc  = roc_auc_score(y_te, prob)
    print(f'\n{"="*40}')
    print(f'Model: {name}')
    print(f'Accuracy : {acc:.4f}  |  F1 : {f1:.4f}  |  AUC : {auc:.4f}')
    print(classification_report(y_te, pred, target_names=['No Flood','Flood']))
    results_inline.append({'Model': name, 'Accuracy': round(acc,4), 'F1 Score': round(f1,4), 'AUC-ROC': round(auc,4)})
    return prob

print('Data ready. Input dim:', input_dim)

In [ ]:
# ── MODEL 1: Deep ANN ──────────────────────────────────────────
ann = Sequential([
    Dense(256, activation='relu', input_shape=(input_dim,)),
    BatchNormalization(), Dropout(0.3),
    Dense(128, activation='relu'),
    BatchNormalization(), Dropout(0.3),
    Dense(64, activation='relu'),
    BatchNormalization(), Dropout(0.2),
    Dense(32, activation='relu'), Dropout(0.2),
    Dense(1, activation='sigmoid')
])
ann.compile(optimizer=Adam(0.001), loss='binary_crossentropy', metrics=['accuracy'])
ann.fit(X_train, y_train, epochs=200, batch_size=32, validation_split=0.2,
        class_weight=class_weight, callbacks=[es], verbose=0)

ann_prob = evaluate('Deep ANN', ann, X_test, y_test)

In [ ]:
# ── MODEL 2: 1D-CNN ────────────────────────────────────────────
X_tr_cnn = X_train.reshape(X_train.shape[0], input_dim, 1)
X_te_cnn = X_test.reshape(X_test.shape[0], input_dim, 1)

cnn = Sequential([
    Conv1D(64, 3, activation='relu', padding='same', input_shape=(input_dim,1)),
    BatchNormalization(), MaxPooling1D(2), Dropout(0.3),
    Conv1D(128, 3, activation='relu', padding='same'),
    BatchNormalization(), MaxPooling1D(2), Dropout(0.3),
    Conv1D(64, 3, activation='relu', padding='same'),
    BatchNormalization(), Dropout(0.2),
    Flatten(),
    Dense(128, activation='relu'), Dropout(0.3),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])
cnn.compile(optimizer=Adam(0.001), loss='binary_crossentropy', metrics=['accuracy'])
cnn.fit(X_tr_cnn, y_train, epochs=200, batch_size=32, validation_split=0.2,
        class_weight=class_weight, callbacks=[es], verbose=0)

cnn_prob = evaluate('1D-CNN', cnn, X_te_cnn, y_test)

In [ ]:
# ── MODEL 3: Bidirectional LSTM ────────────────────────────────
X_tr_lstm = X_train.reshape(X_train.shape[0], input_dim, 1)
X_te_lstm = X_test.reshape(X_test.shape[0], input_dim, 1)

lstm = Sequential([
    Bidirectional(LSTM(128, return_sequences=True), input_shape=(input_dim,1)),
    BatchNormalization(), Dropout(0.3),
    Bidirectional(LSTM(64, return_sequences=True)),
    BatchNormalization(), Dropout(0.3),
    Bidirectional(LSTM(32)),
    BatchNormalization(), Dropout(0.2),
    Dense(64, activation='relu'), Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])
lstm.compile(optimizer=Adam(0.001), loss='binary_crossentropy', metrics=['accuracy'])
lstm.fit(X_tr_lstm, y_train, epochs=200, batch_size=32, validation_split=0.2,
         class_weight=class_weight, callbacks=[es], verbose=0)

lstm_prob = evaluate('BiLSTM', lstm, X_te_lstm, y_test)

In [ ]:
# ── MODEL 4: Autoencoder + Classifier ─────────────────────────
inp = Input(shape=(input_dim,))
enc = Dense(128, activation='relu')(inp)
enc = BatchNormalization()(enc)
enc = Dense(64, activation='relu')(enc)
enc = Dense(32, activation='relu')(enc)
dec = Dense(64, activation='relu')(enc)
dec = Dense(128, activation='relu')(dec)
dec = Dense(input_dim, activation='linear')(dec)

ae = Model(inp, dec)
ae.compile(optimizer=Adam(0.001), loss='mse')
ae.fit(X_train, X_train, epochs=100, batch_size=32, validation_split=0.2,
       callbacks=[EarlyStopping(patience=10, restore_best_weights=True)], verbose=0)

encoder_model = Model(inp, enc)
for l in encoder_model.layers: l.trainable = False

clf_inp = Input(shape=(input_dim,))
x = encoder_model(clf_inp)
x = Dense(64, activation='relu')(x)
x = Dropout(0.3)(x)
x = Dense(32, activation='relu')(x)
out = Dense(1, activation='sigmoid')(x)
clf = Model(clf_inp, out)
clf.compile(optimizer=Adam(0.001), loss='binary_crossentropy', metrics=['accuracy'])
clf.fit(X_train, y_train, epochs=150, batch_size=32, validation_split=0.2,
        class_weight=class_weight, callbacks=[es], verbose=0)

# Fine-tune
for l in clf.layers: l.trainable = True
clf.compile(optimizer=Adam(0.0001), loss='binary_crossentropy', metrics=['accuracy'])
clf.fit(X_train, y_train, epochs=100, batch_size=32, validation_split=0.2,
        class_weight=class_weight,
        callbacks=[EarlyStopping(patience=10, restore_best_weights=True)], verbose=0)

ae_prob = evaluate('Autoencoder+Clf', clf, X_test, y_test)

In [ ]:
# ── MODEL 5: TabNet ────────────────────────────────────────────
try:
    from pytorch_tabnet.tab_model import TabNetClassifier
    import torch
    torch.manual_seed(42)

    X_tr_f = X_train.astype(np.float32)
    X_te_f = X_test.astype(np.float32)
    X_tr_t, X_vl_t, y_tr_t, y_vl_t = train_test_split(X_tr_f, y_train, test_size=0.2,
                                                         random_state=42, stratify=y_train)

    tabnet = TabNetClassifier(n_d=32, n_a=32, n_steps=5, gamma=1.3,
                              seed=42, verbose=0)
    tabnet.fit(X_train=X_tr_t, y_train=y_tr_t,
               eval_set=[(X_vl_t, y_vl_t)], eval_name=['val'],
               eval_metric=['auc'], max_epochs=200, patience=20,
               batch_size=256, virtual_batch_size=128, weights=1)

    tn_pred = tabnet.predict(X_te_f)
    tn_prob = tabnet.predict_proba(X_te_f)[:, 1]
    acc  = accuracy_score(y_test, tn_pred)
    f1   = f1_score(y_test, tn_pred)
    auc  = roc_auc_score(y_test, tn_prob)
    print(f'\nTabNet — Accuracy: {acc:.4f}  |  F1: {f1:.4f}  |  AUC: {auc:.4f}')
    print(classification_report(y_test, tn_pred, target_names=['No Flood','Flood']))
    results_inline.append({'Model': 'TabNet', 'Accuracy': round(acc,4),
                            'F1 Score': round(f1,4), 'AUC-ROC': round(auc,4)})
    tabnet_prob = tn_prob
except ImportError:
    print('TabNet not installed. Run: pip install pytorch-tabnet')
    print('Skipping TabNet...')
    tabnet_prob = None

## Comparison Table

In [ ]:
# Use inline results if available, else use loaded JSON results
if results_inline:
    df_cmp = pd.DataFrame(results_inline).sort_values('F1 Score', ascending=False).reset_index(drop=True)
else:
    df_cmp = df_results.rename(columns={'f1':'F1 Score','accuracy':'Accuracy','model':'Model'})

print('\n' + '='*55)
print('          DL MODEL COMPARISON — BIHAR FLOOD')
print('='*55)
print(df_cmp.to_string(index=False))
print('='*55)

## Bar Chart — Accuracy, F1 & AUC-ROC

In [ ]:
models = df_cmp['Model'].tolist()
x = np.arange(len(models))
width = 0.25

fig, ax = plt.subplots(figsize=(13, 6))

bars1 = ax.bar(x - width,   df_cmp['Accuracy'],  width, label='Accuracy',  color='steelblue')
bars2 = ax.bar(x,           df_cmp['F1 Score'],  width, label='F1 Score',  color='darkorange')

if 'AUC-ROC' in df_cmp.columns:
    bars3 = ax.bar(x + width, df_cmp['AUC-ROC'], width, label='AUC-ROC', color='seagreen')

ax.set_xlabel('Model', fontsize=13)
ax.set_ylabel('Score', fontsize=13)
ax.set_title('DL Model Comparison — Bihar Flood Prediction', fontsize=15, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(models, fontsize=11)
ax.set_ylim(0.5, 1.05)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
ax.legend(fontsize=11)
ax.grid(axis='y', linestyle='--', alpha=0.5)

# Annotate bars
for bar in [*bars1, *bars2, *(bars3 if 'AUC-ROC' in df_cmp.columns else [])]:
    ax.annotate(f'{bar.get_height():.3f}',
                xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 3), textcoords='offset points',
                ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('dl_model_comparison.png', dpi=150)
plt.show()
print('Chart saved as dl_model_comparison.png')

## ROC Curves — All Models

In [ ]:
from sklearn.metrics import RocCurveDisplay

probs = {
    'Deep ANN':          ann_prob,
    '1D-CNN':            cnn_prob,
    'BiLSTM':            lstm_prob,
    'Autoencoder+Clf':   ae_prob,
}
if tabnet_prob is not None:
    probs['TabNet'] = tabnet_prob

fig, ax = plt.subplots(figsize=(9, 7))
for name, prob in probs.items():
    RocCurveDisplay.from_predictions(y_test, prob, name=name, ax=ax)

ax.plot([0,1], [0,1], 'k--', label='Random')
ax.set_title('ROC Curves — All DL Models', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('dl_roc_curves.png', dpi=150)
plt.show()
print('ROC curves saved as dl_roc_curves.png')

## Best Model Summary

In [ ]:
best = df_cmp.iloc[0]
print(f'\n🏆 Best DL Model by F1 Score: {best["Model"]}')
print(f'   Accuracy : {best["Accuracy"]}')
print(f'   F1 Score : {best["F1 Score"]}')
if 'AUC-ROC' in best:
    print(f'   AUC-ROC  : {best["AUC-ROC"]}')